In [1]:
# RQ3a: is the offline phase worth paying for 1 to 10 000 queries? (spec 2, H3a; decision 28 runs the
# session check at 128 MiB x 1 KiB only)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

SESSION = repo / "results/raw/p6-20261113/session"
session = frame.load_session(SESSION, platform)
t = plots.build(df, status, session=session)
cmp_meta = json.loads((SESSION / "comparison.json").read_text())
print(f"grid run {cmp_meta['grid_run']}, session run {cmp_meta['session_run']}, Q = {cmp_meta['queries']},",
      f"{session[frame.KEY].drop_duplicates().shape[0]} cells")

grid run all-20261112_tier1, session run p6-20261113, Q = 100, 96 cells


In [2]:
# Session check (spec 5): per cell one cold offline phase, then Q = 100 consecutive online queries in one process,
# against the grid: computed = median(offline) + Q x median(online). Spec 5 asks for an explanation of every total
# more than 5 % off at Q = 100. Then the relative difference in % at Q = 1, 10 and 100 per arm and quantity.
# comparison.json pairs every grid cell; keep the ones the rq3 tier ran (decision 28).
session = session[session.measured.notna()]
print(session_check.report(session, cmp_meta["queries"]))
rel = session.pivot_table(index=["arm", "quantity"], columns="q", values="rel_diff")
print((100 * rel).round(1).to_string())

arm          db  record quantity        computed       measured     diff
M0       128MiB    1KiB server_s         2.52887        2.46952    -2.3%  (nan)
M0       128MiB    1KiB client_s     0.000511108    0.000464047    -9.2%  FLAG > 5 %
M0       128MiB    1KiB bytes        5.24418e+07    5.24418e+07    +0.0%  (nan)
M0       128MiB    1KiB total_s          2.52938        2.46998    -2.3%  (nan)
M1       128MiB    1KiB server_s         4.52283        4.42085    -2.3%  (nan)
M1       128MiB    1KiB client_s      0.00157137     0.00146363    -6.9%  FLAG > 5 %
M1       128MiB    1KiB bytes        8.07554e+07    8.07554e+07    +0.0%  (nan)
M1       128MiB    1KiB total_s          4.52441        4.42232    -2.3%  (nan)
S0-cpp   128MiB    1KiB server_s         35.0149        35.1234    +0.3%  (nan)
S0-cpp   128MiB    1KiB client_s         1.65875         1.6504    -0.5%  (nan)
S0-cpp   128MiB    1KiB bytes        2.34527e+08    2.34527e+08    +0.0%  (nan)
S0-cpp   128MiB    1KiB total_s      

In [3]:
# H3a: break-even Q from the grid medians (amortise.break_even_Q, closed form): the smallest Q at which arm a's
# amortised cost per query, offline / Q + online, is at most arm b's. None: never. 1: from the first query.
# Price pairs first, then single-server against two-server. 1 KiB records at every size both arms ran.
PAIRS = [("S1", "S0-cpp"), ("S2", "S0-cpp"), ("M1", "M0"), ("S1", "M1"), ("S2", "M1"), ("M1", "S1"), ("M1", "S2")]
med = t.medians
rows = []
for a, b in PAIRS:
    for db in sorted(med.db_bytes.unique()):
        ca, cb = plots._cell(med, a, db, 1 << 10), plots._cell(med, b, db, 1 << 10)
        if ca is not None and cb is not None:
            rows.append({"a": a, "b": b, "db": fmt_size(int(db)), **{k: amortise.break_even_Q(ca, cb, k)
                                                                      for k in ("bytes", "client_s", "server_s", "total_s")}})
print(pd.DataFrame(rows).to_string(index=False))

 a      b     db  bytes  client_s  server_s   total_s
S1 S0-cpp   1MiB    NaN       NaN  505365.0       NaN
S1 S0-cpp  16MiB    NaN       NaN  555463.0       NaN
S1 S0-cpp 128MiB    NaN       NaN  357550.0       NaN
S1 S0-cpp   1GiB    NaN       NaN  384058.0 5574686.0
S2 S0-cpp   1MiB    NaN       NaN       NaN       NaN
S2 S0-cpp  16MiB    NaN       NaN       NaN       NaN
S2 S0-cpp 128MiB    NaN       NaN       NaN       NaN
M1     M0   1MiB    NaN       NaN       NaN       NaN
M1     M0  16MiB    NaN       NaN       NaN       NaN
M1     M0 128MiB    NaN       NaN       NaN       NaN
M1     M0   1GiB    NaN       NaN       NaN       NaN
S1     M1   1MiB  643.0       NaN   22337.0       NaN
S1     M1  16MiB  508.0       NaN    6641.0       NaN
S1     M1 128MiB  715.0       NaN    5837.0   25852.0
S1     M1   1GiB  454.0       NaN    5342.0    7075.0
S2     M1   1MiB 1034.0       NaN       NaN       NaN
S2     M1  16MiB  674.0       NaN   21297.0       NaN
S2     M1 128MiB  892.0     

In [4]:
# Which arm costs least per query at each Q, at 128 MiB x 1 KiB (S3 is per retrieved bit, so it is listed apart):
# amortised bytes and client + server seconds per query from the grid medians.
am = t.amortised[(t.amortised.db_bytes == CELL[0]) & (t.amortised.record_bytes == CELL[1])]
for kind in ("bytes", "total_s"):
    tab = am.pivot_table(index="arm", columns="Q", values=kind)
    print(f"{kind} per query\n{tab.to_string(float_format=lambda v: f'{v:.4g}')}")
    ver = tab.loc[[a for a in ("S1", "S2", "M1") if a in tab.index]]
    print("cheapest arm:", {q: tab[q].idxmin() for q in tab.columns})
    print("cheapest verifiable arm:", {q: ver[q].idxmin() for q in ver.columns}, "\n")
s3 = t.amortised[(t.amortised.arm == "S3") & (t.amortised.db_bytes == CELL[0])]
print("S3 at 128 MiB, one bit per query:\n", s3[["Q", "bytes", "total_s"]].to_string(index=False))

bytes per query
Q          1         10        100       1000      10000
arm                                                     
M0     5.244e+05 5.244e+05 5.244e+05 5.244e+05 5.244e+05
M1     8.076e+05 8.076e+05 8.076e+05 8.076e+05 8.076e+05
S0-cpp 2.237e+08 2.247e+07 2.345e+06 3.328e+05 1.315e+05
S0-go  8.887e+07 8.965e+06 9.745e+05 1.755e+05 9.558e+04
S1     4.993e+08 5.002e+07 5.101e+06 6.083e+05 1.591e+05
S2     6.033e+08 6.045e+07 6.163e+06 7.342e+05 1.914e+05
cheapest arm: {1: 'M0', 10: 'M0', 100: 'M0', 1000: 'S0-go', 10000: 'S0-go'}
cheapest verifiable arm: {1: 'M1', 10: 'M1', 100: 'M1', 1000: 'S1', 10000: 'S1'} 

total_s per query
Q        1       10      100     1000    10000
arm                                           
M0     0.02529 0.02529 0.02529 0.02529 0.02529
M1      0.8218  0.1158 0.04524 0.03818 0.03748
S0-cpp   33.52   3.381  0.3667 0.06534  0.0352
S0-go    10.31   1.045  0.1184 0.02576  0.0165
S1         134   13.43   1.372  0.1662 0.04565
S2       192.5   19.28

In [5]:
# F4 (amortised cost against Q at the largest size where S1, S2 and M1 all ran) and F8 (measured against
# computed totals for Q = 1, 10, 100 and 1 000).
for name in ("F4_amortised", "F8_session"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F[48]_*")))

['F4_amortised.pdf', 'F4_amortised.png', 'F8_session.pdf', 'F8_session.png']
